<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/IIRSI/Project/Step_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Этап 9. Основное API и интеграция — подробная методичка

Эта методичка продолжает Этапы 1–8. Предполагается, что у вас:
- `docker compose ps` — `db`, `redis`, `qdrant`, `ollama` в `healthy`, `app` в `healthy`;
- `/health` возвращает 200, все `services.*.healthy: true`;
- 4 файла из Этапа 5 (`http_client.py`, `hf_llm.py`, `llm_processing.py`, `llm_client.py`) на месте;
- заглушки `_call_llm` в `Critic` и `Editor` удалены (Этап 8.5);
- `perform_translation` пока **не существует** — его создаём здесь;
- 5 таблиц в БД, ≥ 1000 TM, ≥ 500 glossary, Qdrant-коллекция `tm_vectors` заполнена.

Если чего-то нет — вернитесь к Этапу 8.

**Цель этапа:** центральный модуль `perform_translation` с 8 ablation-режимами, 4 translate-роута, админ-CRUD, кэш.

**Недель:** 2.

**Что вы получите в конце этапа:**

- `backend/services/pipeline.py` — центральный модуль (~400 строк): `_MODE_CONFIG`, `build_translation_prompt`, `translate_with_model`, `select_best_variant`, `empty_result`, `_retrieve_context`, `perform_translation`, `safe_perform_translation`, `collect_retrieval_context`, `is_ensemble_language_pair`, `_normalize_model_name`, `_mode_config`.
- `backend/core/models.py` — Pydantic-модели для translate-роутов (расширение из Этапа 2).
- `backend/data/cache.py` — `TranslationCacheManager` с сентинелом `"unknown"` и `hits`-счётчиком.
- `backend/api/routes/translate.py` — 4 эндпоинта: `/translate`, `/translate/multi`, `/translate/batch`, `/translate/ensemble`; семафор на 4.
- `backend/api/routes/admin.py` — CRUD для glossary/TM, импорт CSV, статистика кэша.
- `backend/api/main.py` — регистрация `translate` и `admin` роутеров.
- Интеграционные тесты `test_api.py`.

**Критерии готовности:**

- [ ] `baseline` вызывает только LLM; `glossary` — + глоссарий; `tm_hybrid` — + гибридный RAG; `full` — + агенты.
- [ ] `mode = eval_mode or settings.EVAL_MODE` — дефолт из `.env`.
- [ ] `strict_engine=True` + `engine="ollama"` + fallback → `EngineFallbackTriggeredError` (503).
- [ ] `engine=None` в `get_cached` означает «без фильтра» — кэш срабатывает во второй раз.
- [ ] Кэш только для `domain == "general"` (`CACHEABLE_DOMAIN`).
- [ ] `_translation_semaphore` = 4; 50 текстов через `/batch` не валят Ollama.
- [ ] CSV-импорт идемпотентен.
- [ ] Ensemble не падает при timeout одной модели; при всех `critic_score=None` — первый вариант с warning.
- [ ] `reasoning_log` в `TranslateResponse`.
- [ ] Арабский источник: `strip_arabic_diacritics` до отправки в LLM.
- [ ] Ветка `week-9` создана, коммит сделан.

> **Важно.** Логика pipeline вынесена в `backend/services/pipeline.py`, **не** в `api/routes/translate.py`. Причина: `perform_translation` используется и роутом, и Celery-задачами (Этап 11), и eval-скриптами (Этап 12). Если бы функция жила в роуте — Celery импортировал бы `api.routes.translate`, что тянуло бы `Depends(get_current_user)` и весь DI-граф. Отдельный `services/pipeline.py` — чистый модуль без FastAPI-зависимостей.

---

## 9.0. Подготовка: ветка

```powershell
cd D:\RUNG
git checkout -b week-9
```

Проверьте, что в `pyproject.toml` есть `python-multipart = "^0.0.32"` — нужен для `UploadFile` в `/admin/glossary/import-csv` (FastAPI использует его для multipart-парсинга).

---

## 9.1. `backend/core/models.py` — расширение Pydantic-моделей

**Что это:** в Этапе 2 создан минимальный `models.py` (`TranslationDetails`, `GlossaryEntry`, `TMEntry`, `validate_language_code`, `TranslationEngine`). Сейчас добавляются 10 моделей для translate-роутов.

**Зачем это нужно:** Pydantic-модели для request/response — контракт API. `TranslateRequest` валидирует `text`, языки, engine. `TranslateResponse` описывает, что клиент получит.

Откройте `backend/core/models.py` и **добавьте** следующие модели (не удаляя существующие — `GlossaryEntry`, `TMEntry`, `TranslationDetails`, `validate_language_code`, `TranslationEngine`):

```python
# backend/core/models.py — добавить после существующего TranslationDetails

# ---------------------------------------------------------------------------
# /translate
# ---------------------------------------------------------------------------

class TranslateRequest(BaseModel):
    text: str
    source_lang: str = "en"
    target_lang: str = "ru"
    # If None, the engine is taken from settings (TRANSLATION_ENGINE).
    engine: Optional[TranslationEngine] = None
    # Free-form domain / audience hints passed to the Critic.
    domain: str = "general"
    audience: str = "general"

    @field_validator("source_lang", "target_lang")
    @classmethod
    def validate_lang(cls, v: str) -> str:
        return validate_language_code(v)


class TranslateResponse(BaseModel):
    source_text: str
    source_lang: str
    target_lang: str
    translation: str
    model_used: str
    # Kept for backward compatibility.
    translator_used: str = "ollama"
    # Actual engine used: "ollama" or "huggingface".
    engine_used: Optional[str] = None
    glossary_used: List[GlossaryEntry] = Field(default_factory=list)
    tm_used: List[TMEntry] = Field(default_factory=list)
    validation_passed: Optional[bool] = None
    critic_score: Optional[float] = None
    iterations: Optional[int] = None
    critic_reasoning: Optional[str] = None
    editor_changed: Optional[bool] = None
    critic_adequacy: Optional[float] = None
    critic_fluency: Optional[float] = None
    critic_terminology: Optional[float] = None
    critic_suggestions: Optional[List[str]] = None
    critic_missing_terms: Optional[List[str]] = None
    reasoning_log: Optional[List[Dict[str, Any]]] = None


# ---------------------------------------------------------------------------
# /translate/multi
# ---------------------------------------------------------------------------

class TranslateMultiRequest(BaseModel):
    text: str
    source_lang: str = "en"
    target_langs: List[str]
    engine: Optional[TranslationEngine] = None
    domain: str = "general"
    audience: str = "general"

    @field_validator("source_lang")
    @classmethod
    def validate_source_lang(cls, v: str) -> str:
        return validate_language_code(v)

    @field_validator("target_langs")
    @classmethod
    def validate_target_langs(cls, v: List[str]) -> List[str]:
        for lang in v:
            validate_language_code(lang)
        return v


class TranslateMultiResponse(BaseModel):
    source_text: str
    source_lang: str
    translations: Dict[str, str]
    details: Optional[Dict[str, TranslationDetails]] = None


# ---------------------------------------------------------------------------
# /translate/batch
# ---------------------------------------------------------------------------

class TranslateBatchRequest(BaseModel):
    texts: List[str]
    source_lang: str = "en"
    target_lang: str = "ru"
    engine: Optional[TranslationEngine] = None
    domain: str = "general"
    audience: str = "general"

    @field_validator("source_lang", "target_lang")
    @classmethod
    def validate_lang(cls, v: str) -> str:
        return validate_language_code(v)


class TranslateBatchResponse(BaseModel):
    source_lang: str
    target_lang: str
    translations: List[str]
    details: Optional[List[TranslationDetails]] = None


# ---------------------------------------------------------------------------
# /translate/ensemble
# ---------------------------------------------------------------------------

class EnsembleRequest(BaseModel):
    text: str
    source_lang: str = "en"
    target_lang: str = "ru"
    models: Optional[List[str]] = None
    domain: str = "general"
    audience: str = "general"

    @field_validator("source_lang", "target_lang")
    @classmethod
    def validate_lang(cls, v: str) -> str:
        return validate_language_code(v)


class EnsembleTranslationVariant(BaseModel):
    model: str
    translation: str
    critic_score: Optional[float] = None
    critic_adequacy: Optional[float] = None
    critic_fluency: Optional[float] = None
    critic_terminology: Optional[float] = None
    critic_reasoning: Optional[str] = None


class EnsembleResponse(BaseModel):
    source_text: str
    source_lang: str
    target_lang: str
    best_translation: str
    best_model: str
    best_critic_score: float
    variants: List[EnsembleTranslationVariant]
```

### Разбор ключевых решений

**Почему `translator_used: str = "ollama"` оставлено в `TranslateResponse`.** Это поле было в ранних версиях API; некоторые клиенты его читают. `engine_used: Optional[str]` — фактический движок ("ollama"/"huggingface"). В `translate.py` (9.4) заполняются оба: `translator_used=result["translator"]` ("ollama"/"huggingface"/"cache"/"error"), `engine_used=result["engine_used"]`.

**Почему `engine: Optional[TranslationEngine] = None`, а не строка.** `TranslationEngine = Literal["auto", "ollama", "huggingface"]` (объявлен в начале `models.py`). Pydantic валидирует поле: клиент не может передать `engine="google"` — получит 422. `None` = «взять из `settings.TRANSLATION_ENGINE`».

**Почему `domain` и `audience` — `str = "general"`, а не `Literal`.** Список доменов открытый: `medical`, `legal`, `news`, `scripture`, `hadith`, `prose`, `poetry`, `encyclopedic`, `technical`, `general`. `Critic` (Этап 5) знает только `medical` из `_DOMAIN_RULES`; неизвестный домен просто пропускает доменный тир и использует universal + language. `Literal` ограничил бы API известными доменами — при добавлении нового пришлось бы менять модель.

**Почему `target_langs: List[str]` в `TranslateMultiRequest` без дефолта.** Multi-translate требует явный список. Дефолт `[]` бессмыслен — запрос без языков бесполезен. Pydantic вернёт 422 при отсутствии поля.

**Почему `field_validator` на `target_langs`, а не на `List[Literal]`.** `validate_language_code` бросает `ValueError` с сообщением «Language 'xx' not supported. Supported: [...]». Pydantic ловит `ValueError` и превращает в 422 с этим сообщением. `Literal` вернул бы generic «value is not a valid enumeration member» — менее информативно.

**Почему `EnsembleResponse.best_critic_score: float`, а не `Optional[float]`.** Если все варианты упали — 502 (в `translate.py`), `EnsembleResponse` не создаётся. Если хотя бы один успешен — `best_critic_score` — `v.critic_score or 0.0` → число. `float` без Optional — правильно.

**Почему `EnsembleTranslationVariant` без `failed` поля.** В `variants` попадают только **успешно** прошедшие модели (с `critic_score` или `None`). Упавшие с `Exception` — отбрасываются в `translate.py` с логом `error`, но не включаются в список. `critic_score=None` — только если Critic упал, но сама модель ответила.

**Почему `TranslateResponse.critic_suggestions: Optional[List[str]]`.** Когда `baseline` (нет агентов) — `critic_suggestions=None`. Когда `glossary` — тоже `None`. Когда `full` — список строк (может быть `[]`). `Optional` — правильно для «поля может не быть в ответе».

---

## 9.2. `backend/data/cache.py` — кэш переводов

**Что это:** `TranslationCacheManager` с методами `get_cached`, `save_cache`, `clear_cache`. Сентинел `"unknown"` для опциональных ключевых полей.

**Зачем это нужно:** `/translate` для `baseline`/`glossary`/`tm`/`tm_hybrid` (без агентов) — детерминированный. Для `agents_only`/`tm_hybrid_critic`/`tm_hybrid_editor`/`full` — с `seed=42` и `temperature=0.0/0.3` — тоже детерминированный. Кэш на 6 полей позволяет вернуть точный результат без повторного вызова LLM.

Создайте `backend/data/cache.py`:

```python
# backend/data/cache.py
import logging
from typing import Optional, Tuple

from sqlalchemy import delete, select

from backend.db.models import TranslationCache
from backend.db.session import AsyncSessionLocal

logger = logging.getLogger(__name__)

CacheHit = Tuple[str, str, str]


class TranslationCacheManager:
    """
    Persistent translation cache.

    Cache key:
        (source_lang, target_lang, engine_used, eval_mode,
         model_used, source_text)

    ``engine_used``, ``eval_mode`` and ``model_used`` are stored as
    non-empty strings. When the caller does not supply one on write, the
    sentinel ``"unknown"`` is used — SQL treats ``NULL`` as distinct in
    UNIQUE constraints, so a NULL-able key column would silently allow
    duplicate rows.

    On read (``get_cached``), all three optional key fields follow the
    same rule:

    - ``engine=None``   matches any engine.
    - ``model=None``    matches any model.
    - ``eval_mode=None`` matches any eval mode.
    """

    _UNKNOWN = "unknown"

    @staticmethod
    def _normalize(value: Optional[str]) -> str:
        return (value or TranslationCacheManager._UNKNOWN).strip() \
            or TranslationCacheManager._UNKNOWN

    @staticmethod
    async def get_cached(
        source_text: str,
        source_lang: str,
        target_lang: str,
        engine: Optional[str] = None,
        eval_mode: Optional[str] = None,
        model: Optional[str] = None,
    ) -> Optional[CacheHit]:
        async with AsyncSessionLocal() as session:
            try:
                stmt = select(TranslationCache).where(
                    TranslationCache.source_text == source_text,
                    TranslationCache.source_lang == source_lang,
                    TranslationCache.target_lang == target_lang,
                )
                if eval_mode is not None:
                    stmt = stmt.where(
                        TranslationCache.eval_mode
                        == TranslationCacheManager._normalize(eval_mode)
                    )
                if engine is not None:
                    stmt = stmt.where(TranslationCache.engine_used == engine)
                if model is not None:
                    stmt = stmt.where(
                        TranslationCache.model_used
                        == TranslationCacheManager._normalize(model)
                    )
                stmt = stmt.limit(1)

                entry = (await session.execute(stmt)).scalar_one_or_none()
                if entry is None:
                    return None

                entry.hits = (entry.hits or 0) + 1
                await session.commit()
                return (entry.translation, entry.engine_used, entry.model_used)
            except Exception as e:
                logger.error("Cache get error: %s", e, exc_info=True)
                await session.rollback()
                return None

    @staticmethod
    async def save_cache(
        source_text: str,
        source_lang: str,
        target_lang: str,
        translation: str,
        model_used: str,
        engine: Optional[str] = None,
        eval_mode: Optional[str] = None,
    ) -> None:
        engine_value = TranslationCacheManager._normalize(engine)
        eval_mode_value = TranslationCacheManager._normalize(eval_mode)
        model_value = TranslationCacheManager._normalize(model_used)

        async with AsyncSessionLocal() as session:
            try:
                stmt = select(TranslationCache).where(
                    TranslationCache.source_text == source_text,
                    TranslationCache.source_lang == source_lang,
                    TranslationCache.target_lang == target_lang,
                    TranslationCache.engine_used == engine_value,
                    TranslationCache.eval_mode == eval_mode_value,
                    TranslationCache.model_used == model_value,
                ).limit(1)

                existing = (await session.execute(stmt)).scalar_one_or_none()
                if existing is not None:
                    return

                session.add(TranslationCache(
                    source_text=source_text,
                    source_lang=source_lang,
                    target_lang=target_lang,
                    translation=translation,
                    model_used=model_value,
                    engine_used=engine_value,
                    eval_mode=eval_mode_value,
                ))
                await session.commit()
            except Exception as e:
                logger.error("Cache save error: %s", e, exc_info=True)
                await session.rollback()

    @staticmethod
    async def clear_cache(
        source_lang: Optional[str] = None,
        target_lang: Optional[str] = None,
        eval_mode: Optional[str] = None,
    ) -> int:
        async with AsyncSessionLocal() as session:
            try:
                stmt = delete(TranslationCache)
                if source_lang:
                    stmt = stmt.where(TranslationCache.source_lang == source_lang)
                if target_lang:
                    stmt = stmt.where(TranslationCache.target_lang == target_lang)
                if eval_mode is not None:
                    stmt = stmt.where(TranslationCache.eval_mode == eval_mode)

                result = await session.execute(stmt)
                await session.commit()
                deleted = result.rowcount or 0
                return deleted
            except Exception as e:
                logger.error("Cache clear error: %s", e, exc_info=True)
                await session.rollback()
                return 0


cache_manager = TranslationCacheManager()
```

### Разбор ключевых решений

**Почему `_UNKNOWN = "unknown"` — сентинел.** SQL-стандарт: `NULL` в UNIQUE-индексе не равен `NULL`, то есть две строки с одинаковыми полями, где одно поле `NULL`, **не конфликтуют** — обе вставятся. Если `engine_used = NULL` при `save_cache`, повторный `save_cache` с тем же `NULL` создаст дубликат. С `"unknown"` — UNIQUE работает: `"unknown" == "unknown"` → конфликт.

**Почему `_normalize` вызывается и в `get_cached`, и в `save_cache`.** В `save_cache` — чтобы записать сентинел, если `None`. В `get_cached` — только для `eval_mode` и `model` (не для `engine`). `engine_used` в БД хранится как явное значение (`"ollama"` или `"huggingface"`) после `_normalize`, но при чтении `engine=None` означает «любой» — просто пропускаем фильтр, `_normalize` не нужен.

**Почему `.strip() or ...` в `_normalize`.** Двойная защита: `value="  "` (пробелы) → `.strip()` → `""` → `or _UNKNOWN` → `"unknown"`. Без `.strip()` `"  "` стал бы `"  "` в БД — не равно `"unknown"`.

**Почему `get_cached` не фильтрует по `engine`, если `engine=None`.** `perform_translation` сначала вычисляет `engine_used` (Ollama или HF), потом ищет кэш с `engine=engine_used`. Это конкретное значение, всегда. `engine=None` — только для случаев «а что вообще есть в кэше для этого текста». В `perform_translation` `engine=None` не используется, но API `get_cached` поддерживает — для будущих сценариев (например, eval-скрипт «найти любой кэш для текста»).

**Почему инкремент `hits` при чтении.** Каждое чтение = попадание. `hits` — счётчик. `UPDATE translation_cache SET hits = hits + 1 WHERE id = ?` в рамках `get_cached`. `commit()` — чтобы записалось. Без `commit` — транзакция откатится при выходе из `async with`.

**Почему `except Exception` возвращает `None`, а не бросает.** Кэш — оптимизация. Если SELECT упал (БД недоступна) — лучше вернуть «нет кэша» и продолжить реальный перевод, чем упасть с 500. `perform_translation` пойдёт по slow path (LLM), результат — тот же, но медленнее.

**Почему `save_cache` — `SELECT-then-INSERT`, а не `INSERT ... ON CONFLICT DO NOTHING`.** `TranslationCache.id` — `uuid4`, не детерминированный. UNIQUE — на 6 полях, `ON CONFLICT` требовал бы `index_elements=[все 6]`. `SELECT-then-INSERT` проще и читаемее. Race condition возможен (два процесса одновременно вставляют) — `UNIQUE` поймает `IntegrityError`, `except Exception` его проглотит, `rollback`. Один из двух записей — выиграет.

**Почему `save_cache` с `existing is not None: return`.** Если запись уже есть — ничего не делаем. `hits` не сбрасывается (это делает `get_cached` при чтении). Идемпотентность: повторный `save_cache` — no-op.

**Почему `clear_cache` принимает `eval_mode: Optional[str]`.** Три режима вызова:
- `clear_cache()` — всё.
- `clear_cache(source_lang="en")` — только en-источники.
- `clear_cache(eval_mode="tm_hybrid")` — только tm_hybrid.

`clear_cache(source_lang="en", target_lang="ru", eval_mode="full")` — комбинирует. `None` = «не фильтровать».

**Почему `if source_lang`, а `if eval_mode is not None`.** Разница: `source_lang=""` (пустая строка) → falsy → не фильтруем. `eval_mode=""` → `is not None` True → фильтруем по пустой строке (найдёт `"unknown"`-сентинелы). Пустая строка для `source_lang` — невалидна, для `eval_mode` — возможна (если кто-то сохранил с пустым значением).

---

## 9.3. `backend/services/pipeline.py` — центральный модуль

**Что это:** ядро перевода. `_MODE_CONFIG` (8 режимов), `build_translation_prompt`, `translate_with_model`, `select_best_variant`, `empty_result`, `_retrieve_context`, `perform_translation`, `safe_perform_translation`, `collect_retrieval_context`, `is_ensemble_language_pair`, `_normalize_model_name`, `_mode_config`.

**Зачем это нужно:** `perform_translation` — единственная точка, где собираются все компоненты (RAG + LLM + агенты + кэш + active learning). Роуты (9.4), Celery (11), eval (12) вызывают её. Логика в одном месте — не дублируется.

Создайте `backend/services/pipeline.py`:

```python
# backend/services/pipeline.py
import logging
from typing import Any, Dict, List, Optional, Tuple

from backend.agents.supervisor import Supervisor
from backend.agents.validator import Validator
from backend.core.config import get_settings
from backend.core.constants import CRITIC_PASS_THRESHOLD, KNOWN_LANGUAGES
from backend.core.exceptions import (
    EngineFallbackTriggeredError,
    UnknownEvalModeError,
)
from backend.core.models import EnsembleTranslationVariant
from backend.data.cache import cache_manager
from backend.data.glossary import glossary_manager
from backend.data.translation_memory import tm_manager
from backend.data.translator_manager import translator_manager
from backend.hitl.active_learning import ActiveLearningManager
from backend.retrieval.hybrid_retriever import hybrid_search
from backend.utils.llm_client import call_ollama_with_model
from backend.utils.llm_processing import clean_translation
from backend.utils.text_normalization import strip_arabic_diacritics

logger = logging.getLogger(__name__)

settings = get_settings()

CACHEABLE_DOMAIN = "general"

_MODE_CONFIG: Dict[str, Dict[str, Any]] = {
    "baseline": {
        "use_glossary": False,
        "tm_backend": None,
        "use_critic": False,
        "use_editor": False,
    },
    "glossary": {
        "use_glossary": True,
        "tm_backend": None,
        "use_critic": False,
        "use_editor": False,
    },
    "tm": {
        "use_glossary": True,
        "tm_backend": "bm25",
        "use_critic": False,
        "use_editor": False,
    },
    "tm_hybrid": {
        "use_glossary": True,
        "tm_backend": "hybrid",
        "use_critic": False,
        "use_editor": False,
    },
    "agents_only": {
        "use_glossary": False,
        "tm_backend": None,
        "use_critic": True,
        "use_editor": True,
    },
    "tm_hybrid_critic": {
        "use_glossary": True,
        "tm_backend": "hybrid",
        "use_critic": True,
        "use_editor": False,
    },
    "tm_hybrid_editor": {
        "use_glossary": True,
        "tm_backend": "hybrid",
        "use_critic": False,
        "use_editor": True,
    },
    "full": {
        "use_glossary": True,
        "tm_backend": "hybrid",
        "use_critic": True,
        "use_editor": True,
    },
}

_ALL_MODES: Tuple[str, ...] = tuple(_MODE_CONFIG.keys())


def _normalize_model_name(name: Optional[str]) -> str:
    return (name or "").replace("ollama/", "").strip() or "unknown"


def _mode_config(mode: str) -> Dict[str, Any]:
    if mode not in _MODE_CONFIG:
        raise UnknownEvalModeError(
            f"Unknown eval_mode='{mode}'",
            context={
                "mode": mode,
                "supported_modes": list(_ALL_MODES),
            },
        )
    return _MODE_CONFIG[mode]


def build_translation_prompt(
    text: str,
    source_lang: str,
    target_lang: str,
    glossary_entries: List,
    tm_entries: List,
    domain: str = "general",
) -> str:
    prompt = (
        f"Translate the following text from {source_lang} to {target_lang}. "
        "IMPORTANT: Your response must contain ONLY the translation, with NO additional text, "
        "NO explanations, NO commentary, and NO meta-remarks. "
        "Do NOT add phrases like 'Translation:', 'Here is the translation', or any other text.\n"
    )
    if domain and domain != "general":
        prompt += f"Domain: {domain}. Prefer domain-standard terminology.\n"
    if glossary_entries:
        prompt += "\nGlossary:\n" + "\n".join(
            f"{e.en} → {e.tgt}" for e in glossary_entries
        )
    if tm_entries:
        prompt += "\nPast translations (by relevance):\n"
        for entry in tm_entries:
            score = getattr(entry, "score", None)
            score_suffix = (
                f" (score: {score:.2f})" if score is not None else ""
            )
            prompt += (
                f"{source_lang}: {entry.en}\n"
                f"{target_lang}: {entry.tgt}{score_suffix}\n\n"
            )
    prompt += f"\nText to translate ({source_lang}): {text}\n"
    prompt += f"Translation ({target_lang}):"
    return prompt


async def translate_with_model(
    text: str,
    source_lang: str,
    target_lang: str,
    model_name: str,
    glossary_entries: List,
    tm_entries: List,
    domain: str = "general",
) -> str:
    prompt = build_translation_prompt(
        text, source_lang, target_lang, glossary_entries, tm_entries, domain
    )
    result = await call_ollama_with_model(prompt, model_name)
    return result


def select_best_variant(
    variants: List[EnsembleTranslationVariant],
) -> EnsembleTranslationVariant:
    max_score = max((v.critic_score or 0.0) for v in variants)

    if max_score == 0.0 and all(v.critic_score is None for v in variants):
        logger.warning("Ensemble: no variant was scored by the critic; using first")
        return variants[0]

    tied = [v for v in variants if (v.critic_score or 0.0) == max_score]
    if len(tied) == 1:
        return tied[0]

    logger.info(
        "Ensemble: %d variants tied at score %s; breaking tie by length",
        len(tied),
        max_score,
    )
    return max(tied, key=lambda v: len(v.translation or ""))


def empty_result(
    engine: Optional[str] = None,
    translator: str = "cache",
    model: Optional[str] = None,
) -> Dict[str, Any]:
    return {
        "translation": "",
        "translation_raw": "",
        "translation_after_agents": None,
        "glossary_used": [],
        "tm_used": [],
        "validation_passed": None,
        "critic_score": None,
        "critic_reasoning": None,
        "editor_changed": False,
        "iterations": 0,
        "critic_adequacy": None,
        "critic_fluency": None,
        "critic_terminology": None,
        "critic_suggestions": [],
        "critic_missing_terms": [],
        "cached": False,
        "translator": translator,
        "engine_used": engine,
        "model_used": model,
        "reasoning_log": [],
    }


async def _retrieve_context(
    text: str,
    source_lang: str,
    target_lang: str,
    mode_config: Dict[str, Any],
) -> Tuple[List[Any], List[Any]]:
    glossary_entries: List[Any] = []
    tm_entries: List[Any] = []

    if mode_config["use_glossary"]:
        glossary_entries = await glossary_manager.get_entries(
            text, source_lang, target_lang
        )

    backend = mode_config["tm_backend"]
    if backend == "bm25":
        tm_entries = await tm_manager.search(
            text, source_lang, target_lang, top_n=5
        )
    elif backend == "hybrid":
        tm_entries = await hybrid_search(
            text, source_lang, target_lang, top_n=5
        )

    return glossary_entries, tm_entries


async def perform_translation(
    text: str,
    source_lang: str,
    target_lang: str,
    engine: Optional[str] = None,
    eval_mode: Optional[str] = None,
    domain: str = "general",
    audience: str = "general",
    strict_engine: bool = False,
) -> Dict[str, Any]:
    if source_lang == "ar":
        stripped = strip_arabic_diacritics(text)
        if stripped != text:
            text = stripped

    mode = eval_mode or settings.EVAL_MODE
    mode_config = _mode_config(mode)

    requested_engine = engine or settings.TRANSLATION_ENGINE
    translator_func = await translator_manager.get_translator_for_engine(
        requested_engine, source_lang, target_lang
    )
    engine_used = "huggingface" if translator_func is not None else "ollama"

    if strict_engine and requested_engine == "ollama" and engine_used != "ollama":
        raise EngineFallbackTriggeredError(
            "Strict engine='ollama' requested but HF fallback was selected",
            context={
                "requested_engine": requested_engine,
                "engine_used": engine_used,
                "source_lang": source_lang,
                "target_lang": target_lang,
            },
        )

    model_used = _normalize_model_name(
        settings.HF_TRANSLATION_MODEL
        if engine_used == "huggingface"
        else settings.LLM_MODEL
    )

    cache_allowed = domain == CACHEABLE_DOMAIN

    if cache_allowed:
        cached = await cache_manager.get_cached(
            text, source_lang, target_lang,
            engine=engine_used, eval_mode=mode, model=model_used,
        )
        if cached is not None:
            cached_translation, cached_engine, cached_model = cached
            result = empty_result(
                engine=cached_engine, translator="cache", model=cached_model,
            )
            result["translation"] = cached_translation
            result["critic_reasoning"] = "Cached translation"
            result["cached"] = True
            return result

    glossary_entries, tm_entries = await _retrieve_context(
        text, source_lang, target_lang, mode_config
    )

    use_agents = mode_config["use_critic"] or mode_config["use_editor"]

    state: Dict[str, Any] = {}

    try:
        if translator_func is None:
            prompt = build_translation_prompt(
                text, source_lang, target_lang,
                glossary_entries, tm_entries, domain,
            )
            translation = await call_ollama_with_model(prompt, settings.LLM_MODEL)
            translator_used = "ollama"
        else:
            translation = await translator_func(text, source_lang, target_lang)
            translator_used = "huggingface"
    except Exception as e:
        logger.error("Translation engine error: %s", e, exc_info=True)
        raise

    if use_agents:
        supervisor = Supervisor(max_iterations=3, engine=settings.AGENT_ENGINE)
        state = {
            "source_text": text,
            "source_lang": source_lang,
            "target_lang": target_lang,
            "translation": translation,
            "glossary_used": glossary_entries,
            "tm_used": tm_entries,
            "engine": settings.AGENT_ENGINE,
            "domain": domain,
            "audience": audience,
            "use_critic": mode_config["use_critic"],
            "use_editor": mode_config["use_editor"],
        }
        state = await supervisor.process(state)
        final_translation = state.get("translation", translation)
        validation_passed = state.get("validation_passed")
        critic_score = state.get("critic_score")
        critic_reasoning = state.get("critic_reasoning")
        editor_changed = state.get("editor_changed")
        iterations = state.get("supervisor_iterations")
        reasoning_log = state.get("reasoning_log", [])

        if (
            settings.ACTIVE_LEARNING_ENABLED
            and critic_score is not None
            and (critic_score < CRITIC_PASS_THRESHOLD or not validation_passed)
        ):
            await ActiveLearningManager.add_uncertain_example(
                source_text=text,
                source_lang=source_lang,
                target_lang=target_lang,
                machine_translation=final_translation,
                critic_score=critic_score,
                validation_passed=validation_passed,
            )
    else:
        validator = Validator(engine=settings.AGENT_ENGINE)
        validator_state = await validator.process({
            "source_text": text,
            "source_lang": source_lang,
            "target_lang": target_lang,
            "translation": translation,
            "glossary_used": glossary_entries,
        })
        validation_passed = validator_state.get("validation_passed")

        final_translation = translation
        critic_score = None
        critic_reasoning = None
        editor_changed = None
        iterations = 0
        reasoning_log = []

    final_clean = clean_translation(final_translation)
    if final_clean and len(final_clean) > 1:
        translation_to_return = final_clean
    else:
        translation_to_return = final_translation

    if cache_allowed:
        await cache_manager.save_cache(
            text, source_lang, target_lang, translation_to_return,
            model_used, engine=engine_used, eval_mode=mode,
        )

    return {
        "translation": translation_to_return,
        "translation_raw": translation,
        "translation_after_agents": final_translation if use_agents else None,
        "glossary_used": glossary_entries,
        "tm_used": tm_entries,
        "validation_passed": validation_passed,
        "critic_score": critic_score,
        "critic_reasoning": critic_reasoning,
        "editor_changed": editor_changed,
        "iterations": iterations,
        "critic_adequacy": state.get("critic_adequacy"),
        "critic_fluency": state.get("critic_fluency"),
        "critic_terminology": state.get("critic_terminology"),
        "critic_suggestions": state.get("critic_suggestions"),
        "critic_missing_terms": state.get("critic_missing_terms"),
        "cached": False,
        "translator": translator_used,
        "engine_used": engine_used,
        "model_used": model_used,
        "reasoning_log": reasoning_log,
    }


async def safe_perform_translation(
    text: str,
    source_lang: str,
    target_lang: str,
    engine: Optional[str] = None,
    eval_mode: Optional[str] = None,
    domain: str = "general",
    audience: str = "general",
    strict_engine: bool = False,
) -> Tuple[bool, Dict[str, Any]]:
    try:
        result = await perform_translation(
            text, source_lang, target_lang,
            engine=engine, eval_mode=eval_mode,
            domain=domain, audience=audience,
            strict_engine=strict_engine,
        )
        return True, result
    except Exception as e:
        logger.error(
            "Parallel translation failed for '%s...' (%s->%s): %s",
            text[:50], source_lang, target_lang, e,
            exc_info=True,
        )
        placeholder = empty_result(translator="error")
        placeholder["critic_reasoning"] = f"Translation failed: {e}"
        return False, placeholder


async def collect_retrieval_context(
    text: str,
    source_lang: str,
    target_lang: str,
    mode: str,
) -> Tuple[List[Any], List[Any]]:
    mode_config = _mode_config(mode)
    return await _retrieve_context(text, source_lang, target_lang, mode_config)


def is_ensemble_language_pair(source_lang: str, target_lang: str) -> bool:
    return source_lang in KNOWN_LANGUAGES and target_lang in KNOWN_LANGUAGES
```

### Разбор ключевых решений

**Почему `CACHEABLE_DOMAIN = "general"`, а не `"medical"` или список.** Кэш хранит результат, полученный с конкретным `prompt` (с глоссарием/TM/agent-pipeline). Для `domain != "general"` в промпт попадает `"Domain: medical. Prefer domain-standard terminology."` — Critic мог поставить другую оценку. `Editor` мог внести другие правки. Кэшировать `medical` результат и вернуть его на `general` запрос — значит отдать `"пациент"` вместо `"больной"`. Только `general` — безопасно.

**Почему `_normalize_model_name` — `(name or "").replace("ollama/", "").strip() or "unknown"`.** Три преобразования:
1. `None` → `""`.
2. `"ollama/qwen2.5:7b"` → `"qwen2.5:7b"`.
3. `""` (после strip) → `"unknown"`.

Результат используется как `model_used` в кэше — должен быть non-empty (сентинел для UNIQUE).

**Почему `_mode_config` бросает `UnknownEvalModeError`, а не `KeyError`.** `UnknownEvalModeError` (Этап 2) → 400 с `code: "unknown_eval_mode"` и `context: {mode, supported_modes}`. Клиент сразу видит, какие режимы валидны.

**Почему `build_translation_prompt` содержит `"Do NOT add phrases like 'Translation:'"`.** Ollama/qwen регулярно добавляет префикс. `clean_translation` (Этап 8) убирает постфактум, но лучше в промпте явно запретить — меньше грязи в `clean_translation`, выше точность.

**Почему `if domain and domain != "general"` — блок `Domain:`.** Для `general` — не пишем, чтобы не засорять промпт. Для `medical` — пишем, LLM учитывает.

**Почему `getattr(entry, "score", None)` в `build_translation_prompt`, а не `entry.score`.** `entry` — `TMEntry` или dict. `TMEntry.score: Optional[float]`. `getattr` работает для обоих. `entry.score` упал бы на `dict` с `AttributeError`.

**Почему `score_suffix = f" (score: {score:.2f})" if score is not None else ""`.** BM25/hybrid возвращают `score`. `tm_manager.list_all` (raw read из БД) — `score=None`. Показываем score, если есть, — LLM видит «насколько релевантен» пример. Без score — пример подаётся как есть.

**Почему `translate_with_model` использует `call_ollama_with_model`, а `perform_translation` для Ollama — тоже.** Дублирование? Нет: `translate_with_model` — короткая обёртка (промпт + вызов) для ensemble, где модели разные. `perform_translation` — полный flow с кэшем, retrieval, агентами. Ensemble вызывает `translate_with_model` для каждой модели параллельно, потом Critic. Не то же самое, что `perform_translation` (который делает всё через Supervisor).

**Почему `select_best_variant` — два условия для «все None».** `max_score = max((v.critic_score or 0.0) ...)` — если все None, `max_score = 0.0`. Проверка `max_score == 0.0 and all(v.critic_score is None)` — **все** именно None, не «есть нули». Если есть настоящий 0.0 (Critic вернул 0) — не «все None», идём в tie-break по длине. Edge case: Critic вернул 0.0 для всех — tie at 0.0, break by length. Reference код логически правильно обрабатывает.

**Почему tie-break по `len(v.translation or "")`.** Не по score (они равны). Не по модели (нет приоритета). Длина — эвристика «длиннее = полнее» (для перевода — обычно правда, короткий перевод часто упускает часть смысла). Спорно, но детерминированно.

**Почему `select_best_variant` — sync (не async).** Работает с уже готовыми `EnsembleTranslationVariant`. Никаких `await`. Вызывается из `translate.py` после `await` на всех Critic'ах.

**Почему `empty_result` — единая точка для «пустого» результата.** Используется в трёх местах:
1. Кэш-hit: `empty_result(engine=cached_engine, translator="cache", model=cached_model)`, потом переопределяем `translation` и `critic_reasoning`.
2. `safe_perform_translation` при ошибке: `empty_result(translator="error")`, потом `critic_reasoning`.
3. Eval-скрипт (Этап 12) при ошибке — тоже.

Возвращает все поля, которые `TranslateResponse` ожидает. `validation_passed: None` — «не проверяли». `cached: False` — по умолчанию.

**Почему `empty_result` не принимает `translation` параметром.** Кэш-hit устанавливает `translation` после создания. Ошибка — пустая строка. Не «перегрузка параметром», а явное присваивание — понятнее.

**Почему `_retrieve_context` возвращает `List[Any]`, а не `List[GlossaryEntry]`.** `glossary_manager.get_entries` возвращает `List[GlossaryEntry]`. `tm_manager.search` / `hybrid_search` — `List[TMEntry]`. Type alias `Any` — потому что в разных режимах разное. `build_translation_prompt` работает с обоими (через `e.en`/`e.tgt`).

**Почему `if mode_config["use_glossary"]` — блок в `_retrieve_context`.** Для `baseline` и `agents_only` глоссарий не нужен (`use_glossary: False`) — не делаем запрос. Экономия: `glossary_manager.get_entries` делает `ILIKE '%word%'` — на больших таблицах медленно.

**Почему `backend = mode_config["tm_backend"]; if backend == "bm25"` — две ветки.** `tm` — BM25-only. `tm_hybrid` и все `tm_hybrid_*` — hybrid. `baseline`/`glossary`/`agents_only` — `tm_backend: None` — обе ветки пропускаются, `tm_entries = []`.

**Почему в `perform_translation` сначала `strip_arabic_diacritics`, потом всё остальное.** NLLB и qwen ломаются на Quranic marks (Этап 2: `_TOKENIZER_ARTEFACT_RE`). Strip до кэша и до LLM — `text` в кэше и в БД тоже без marks. Если бы делали после — кэш хранил бы «грязный» текст, а LLM получал «грязный» (или делали бы strip вторично перед вызовом — дублирование).

**Почему `requested_engine = engine or settings.TRANSLATION_ENGINE`.** `engine` (из запроса) приоритетнее. `None` → `.env`. `Settings.validate_engine` (Этап 2) уже валидировал `TRANSLATION_ENGINE`. `engine` из `TranslateRequest.engine: Optional[TranslationEngine]` — Pydantic тоже валидировал. Двойная защита.

**Почему `strict_engine` проверяет **только** `requested_engine == "ollama"`.** Если клиент запросил `engine="auto"` и получил HF — это норма (auto-политика). Если `engine="huggingface"` — получил HF — норма. `strict_engine` имеет смысл только для `engine="ollama"`: «я явно хочу Ollama, не подменяй на HF». Если Ollama недоступна — `EngineFallbackTriggeredError` (503), не тихий fallback.

**Почему `raise EngineFallbackTriggeredError` **до** кэша и **до** retrieval.** Если strict и fallback — пайплайн вообще не имеет смысла (получим HF-результат, который клиент не хочет). Падаем сразу — экономим время.

**Почему `model_used = _normalize_model_name(...)` с тернарником.** Если `engine_used == "huggingface"` — `HF_TRANSLATION_MODEL`. Иначе — `LLM_MODEL`. Это то, что реально использовалось. Сохраняется в кэш как `model_used` — если пользователь сменил `LLM_MODEL`, старый кэш не подойдёт (model_used разный → нет hit'а).

**Почему `cache_allowed = domain == CACHEABLE_DOMAIN`.** Одна переменная, используется дважды: для `get_cached` и `save_cache`. Логика «кэшировать только general» — в одном месте.

**Почему кэш-проверка **до** `_retrieve_context`.** Кэш-hit не требует retrieval — сразу возвращаем. Экономия: не делаем `glossary_manager.get_entries` и `hybrid_search` (медленные).

**Почему кэш-hit возвращает `empty_result(engine=cached_engine, ...)` с полями `None`.** Мы не перезапускаем Critic/Validator на кэше — слишком дорого. `validation_passed: None`, `critic_score: None` — честно «не проверяли в этом запросе». `translation` — точный результат из прошлого запроса. `critic_reasoning = "Cached translation"` — пометка для UI.

**Почему `use_agents = mode_config["use_critic"] or mode_config["use_editor"]`.** True для `agents_only`, `tm_hybrid_critic`, `tm_hybrid_editor`, `full`. False для `baseline`, `glossary`, `tm`, `tm_hybrid`. Если False — не создаём `Supervisor` (он бы создал `Critic`/`Editor` через `_get_*`, потратив ресурсы впустую).

**Почему state для Supervisor содержит `use_critic`/`use_editor` из `mode_config`, а не из `mode`.** Режим `full` → оба True. Режим `tm_hybrid_critic` → `use_critic: True, use_editor: False`. Supervisor читает эти флаги из state (Этап 5). Если бы передавали `mode` — Supervisor должен был бы сам разбираться с маппингом. Проще: caller (pipeline) уже знает из `_MODE_CONFIG`, передаёт флаги.

**Почему `translator_func is None` → `call_ollama_with_model(prompt, settings.LLM_MODEL)`.** `translator_manager.get_translator_for_engine` возвращает:
- `None` — Ollama (использовать `call_ollama_with_model`).
- `Callable` — HF (`TranslatorHuggingFace.translate`).

Для Ollama — строим промпт (с глоссарием/TM), вызываем `call_ollama_with_model`. Для HF — `translator_func(text, src, tgt)` — HF-модель (NLLB) не принимает промпт с глоссарием (это Seq2Seq, не instruction-following). Retrieval-контекст для HF **не используется** — это правильно: NLLB не умеет «вот глоссарий, переведи с учётом».

**Почему `translator_used = "ollama"` / `"huggingface"` — явно.** `engine_used` вычислен из `translator_func is not None`, но `translator_used` — что реально вызвано. Для кэш-hit `translator = "cache"`. Для ошибки — `"error"`. Значения: `"ollama"`, `"huggingface"`, `"cache"`, `"error"`. Клиент различает.

**Почему `except Exception: logger.error(...); raise`.** `raise` — пробрасываем наверх. `translate.py` (9.4) через `_http_error_from_transport` развернёт в 502/504/500 в зависимости от типа. Логируем `exc_info=True` — полный стектрейс в лог для отладки.

**Почему state для Supervisor и state для Validator-only разные.** Для `use_agents=True` — полный state (source, translation, lang, glossary, tm, engine, domain, audience, use_critic, use_editor). Supervisor использует всё. Для `use_agents=False` — минимальный state (source, translation, lang, glossary) — только для Validator. Validator не читает tm, engine, domain, audience.

**Почему `validation_passed` в Validator-only режиме может быть `True`/`False`.** `Validator.process` (Этап 5) прогоняет 7 проверок, возвращает `validation_passed`. Даже без Critic/Editor — базовая проверка. Если `validation_passed=False` — клиент знает, что перевод формально плохой (эхо, пустой, скрипт-микс), даже без LLM-оценки.

**Почему Active Learning в `use_agents=True` — но не в `use_agents=False`.** В baseline нет `critic_score` (`None`). Условие `critic_score is not None` — `None` отсеивает. Но даже без проверки `critic_score is not None` — `critic_score < CRITIC_PASS_THRESHOLD` с `critic_score=None` упал бы с `TypeError`. Условие `is not None` — обязательно.

**Почему Active Learning **не** в ветке `else` (baseline).** В baseline нет `critic_score`, условие `critic_score is not None` не выполнится. Плюс — Active Learning требует «машинный перевод + оценка», в baseline оценки нет. Логически — только для modes с Critic.

**Почему `clean_translation(final_translation)` даже для кэш-результата... нет, не для кэш-результата.** Кэш возвращает `translation_to_return = cached_translation` — уже чистый (сохранён после `clean_translation` при первом сохранении). `clean_translation` применяется только к `final_translation` (результат LLM/Editor). Двойная чистка не нужна.

**Почему `if final_clean and len(final_clean) > 1`.** Два условия:
- `final_clean` — не пустая строка (после `clean_translation`).
- `len(final_clean) > 1` — не один символ (например, `clean_translation` вернул `.` — не перевод).

Если не прошло — `translation_to_return = final_translation` (сырой, но хоть какой-то). Лучше сырой, чем пустой.

**Почему `cache_manager.save_cache` **в самом конце**.** Сохраняем после `clean_translation` — кэш хранит финальный (чистый) результат. Если сохранить до — в кэш попал бы сырой с префиксами.

**Почему возвращаемый dict содержит `translation_raw` и `translation_after_agents`.** `translation_raw` — сырой выход LLM (до `clean_translation`). `translation_after_agents` — после Supervisor (до `clean_translation`). Полезно для eval (Этап 12): можно сравнить, что дала LLM, что сделали агенты, что ушло клиенту. `translation` — что ушло клиенту (после `clean_translation`).

**Почему `translation_after_agents = final_translation if use_agents else None`.** В `use_agents=False` — `final_translation = translation` (то же, что `translation_raw`), смысла дублировать нет → `None`. В `use_agents=True` — `final_translation` может отличаться от `translation` (Editor внёс правки) → сохраняем.

**Почему `reasoning_log` в ответе.** Клиент (Streamlit) показывает шаги Supervisor/Critic/Editor — прозрачность. В `use_agents=False` — `[]`. В кэш-hit — `[]`.

**Почему `safe_perform_translation` возвращает `(bool, dict)`, а не `dict`.** `perform_translation` может бросить (LLM timeout, Qdrant down, `UnknownEvalModeError`). `safe_` ловит, возвращает `(False, placeholder)`. Caller (`/translate/multi`, `/batch`) итерирует `zip(request.target_langs, outcomes)` или `for ok, result in outcomes` — считает failures. `(bool, dict)` — простая структура.

**Почему `placeholder = empty_result(translator="error")`.** То же, что кэш-hit, но с `translator="error"`, `critic_reasoning = "Translation failed: {e}"`. Все поля на месте — `_build_details` в роуте не упадёт на `KeyError`.

**Почему `strict_engine=False` дефолт в `safe_perform_translation`.** `safe_` используется в `/translate/multi` и `/batch`, где strict_engine не передаётся (роуты не принимают). Для eval (Этап 12) `run_eval.py` передаёт `strict_engine=True` напрямую в `perform_translation`. `safe_perform_translation` с `strict_engine=True` — тоже работает (пробрасывает).

**Почему `collect_retrieval_context` — отдельная функция, а не прямо `_retrieve_context`.** `_retrieve_context` — приватная, принимает `mode_config`. `collect_retrieval_context` — публичная, принимает `mode` (строку). Используется в `/translate/ensemble` (9.4) — где нужен retrieval без полного pipeline. Обёртка `mode → mode_config → _retrieve_context`.

**Почему `is_ensemble_language_pair` — простая функция.** `src in KNOWN_LANGUAGES and tgt in KNOWN_LANGUAGES`. Для ensemble — оба языка должны быть Ollama-поддерживаемыми. HF-path для ensemble не реализован (NLLB не умеет в промпт с глоссарием, а ensemble без глоссария — просто N×NLLB-перевод, бессмысленно).

**Почему `settings` — module-level `get_settings()`.** Один раз при импорте. `settings.LLM_MODEL`, `settings.HF_TRANSLATION_MODEL`, `settings.AGENT_ENGINE`, `settings.EVAL_MODE`, `settings.TRANSLATION_ENGINE`, `settings.ACTIVE_LEARNING_ENABLED` — читаются на каждом запросе. В тестах — `monkeypatch.setenv` + `importlib.reload(backend.services.pipeline)`.

**Почему `_ALL_MODES` — tuple, а не list.** Константа. `tuple` — неизменяемый. `list(_ALL_MODES)` в `UnknownEvalModeError.context` создаёт копию для JSON-сериализации.

---

## 9.4. `backend/api/routes/translate.py` — 4 эндпоинта

**Что это:** `/translate`, `/translate/multi`, `/translate/batch`, `/translate/ensemble`. Семафор на 4, обработка transport-ошибок.

**Зачем это нужно:** HTTP-слой над `perform_translation`. Собирает `TranslateResponse` из dict'а. Обрабатывает `httpx`-исключения в правильные HTTP-статусы. Семафор ограничивает параллелизм к Ollama.

Создайте `backend/api/routes/translate.py`:

```python
# backend/api/routes/translate.py
import asyncio
import logging
from typing import Dict, List

import httpx
from fastapi import APIRouter, Depends, HTTPException

from backend.agents.critic import Critic
from backend.api.deps import get_current_user
from backend.core.config import get_settings
from backend.core.exceptions import RUNGException
from backend.core.models import (
    EnsembleRequest, EnsembleResponse, EnsembleTranslationVariant,
    TranslateBatchRequest, TranslateBatchResponse,
    TranslateMultiRequest, TranslateMultiResponse,
    TranslateRequest, TranslateResponse, TranslationDetails,
)
from backend.data.cache import cache_manager
from backend.db.models import User
from backend.services.pipeline import (
    CACHEABLE_DOMAIN, collect_retrieval_context, is_ensemble_language_pair,
    perform_translation, safe_perform_translation, select_best_variant,
    translate_with_model,
)
from backend.utils.llm_processing import clean_translation

logger = logging.getLogger(__name__)
router = APIRouter()
settings = get_settings()

_MAX_CONCURRENT_TRANSLATIONS = 4
_translation_semaphore = asyncio.Semaphore(_MAX_CONCURRENT_TRANSLATIONS)


def _http_error_from_transport(e: Exception, user_email: str) -> HTTPException:
    if isinstance(e, (httpx.TimeoutException, httpx.ReadTimeout)):
        logger.error("Upstream timeout for user=%s", user_email)
        return HTTPException(status_code=504, detail="Upstream timeout")
    if isinstance(e, httpx.ConnectError):
        logger.error("Upstream connection error for user=%s", user_email)
        return HTTPException(status_code=502, detail="Upstream service unavailable")
    if isinstance(e, httpx.HTTPStatusError):
        logger.error(
            "Upstream HTTP error for user=%s: %s",
            user_email, e.response.status_code,
        )
        return HTTPException(
            status_code=502,
            detail=f"Upstream error: {e.response.status_code}",
        )
    logger.error("Unexpected error for user=%s: %s", user_email, e, exc_info=True)
    return HTTPException(status_code=500, detail=str(e))


def _build_details(result: Dict) -> TranslationDetails:
    return TranslationDetails(
        glossary_used=result["glossary_used"],
        tm_used=result["tm_used"],
        critic_score=result["critic_score"],
        validation_passed=result["validation_passed"],
        critic_adequacy=result["critic_adequacy"],
        critic_fluency=result["critic_fluency"],
        critic_terminology=result["critic_terminology"],
        critic_suggestions=result["critic_suggestions"],
        critic_missing_terms=result["critic_missing_terms"],
        reasoning_log=result.get("reasoning_log"),
        engine_used=result.get("engine_used"),
    )


async def _bounded_translate(*args, **kwargs):
    async with _translation_semaphore:
        return await safe_perform_translation(*args, **kwargs)


async def _bounded_translate_with_model(*args, **kwargs):
    async with _translation_semaphore:
        return await translate_with_model(*args, **kwargs)


@router.post("/translate", response_model=TranslateResponse)
async def translate(
    request: TranslateRequest,
    current_user: User = Depends(get_current_user),
):
    try:
        result = await perform_translation(
            request.text,
            request.source_lang,
            request.target_lang,
            engine=request.engine,
            domain=request.domain,
            audience=request.audience,
        )
        return TranslateResponse(
            source_text=request.text,
            source_lang=request.source_lang,
            target_lang=request.target_lang,
            translation=result["translation"],
            model_used=result["model_used"],
            translator_used=result["translator"],
            engine_used=result["engine_used"],
            glossary_used=result["glossary_used"],
            tm_used=result["tm_used"],
            validation_passed=result["validation_passed"],
            critic_score=result["critic_score"],
            iterations=result["iterations"],
            critic_reasoning=result["critic_reasoning"],
            editor_changed=result["editor_changed"],
            critic_adequacy=result["critic_adequacy"],
            critic_fluency=result["critic_fluency"],
            critic_terminology=result["critic_terminology"],
            critic_suggestions=result["critic_suggestions"],
            critic_missing_terms=result["critic_missing_terms"],
            reasoning_log=result.get("reasoning_log"),
        )
    except RUNGException:
        raise
    except HTTPException:
        raise
    except Exception as e:
        raise _http_error_from_transport(e, current_user.email)


@router.post("/translate/multi", response_model=TranslateMultiResponse)
async def translate_multi(
    request: TranslateMultiRequest,
    current_user: User = Depends(get_current_user),
):
    try:
        tasks = [
            _bounded_translate(
                request.text,
                request.source_lang,
                target_lang,
                engine=request.engine,
                domain=request.domain,
                audience=request.audience,
            )
            for target_lang in request.target_langs
        ]
        outcomes = await asyncio.gather(*tasks)

        translations: Dict[str, str] = {}
        details: Dict[str, TranslationDetails] = {}

        for target_lang, (ok, result) in zip(request.target_langs, outcomes):
            translations[target_lang] = result["translation"]
            details[target_lang] = _build_details(result)

        return TranslateMultiResponse(
            source_text=request.text,
            source_lang=request.source_lang,
            translations=translations,
            details=details,
        )
    except RUNGException:
        raise
    except HTTPException:
        raise
    except Exception as e:
        raise _http_error_from_transport(e, current_user.email)


@router.post("/translate/batch", response_model=TranslateBatchResponse)
async def translate_batch(
    request: TranslateBatchRequest,
    current_user: User = Depends(get_current_user),
):
    try:
        tasks = [
            _bounded_translate(
                text,
                request.source_lang,
                request.target_lang,
                engine=request.engine,
                domain=request.domain,
                audience=request.audience,
            )
            for text in request.texts
        ]
        outcomes = await asyncio.gather(*tasks)

        translations: List[str] = []
        details: List[TranslationDetails] = []

        for ok, result in outcomes:
            translations.append(result["translation"])
            details.append(_build_details(result))

        return TranslateBatchResponse(
            source_lang=request.source_lang,
            target_lang=request.target_lang,
            translations=translations,
            details=details,
        )
    except RUNGException:
        raise
    except HTTPException:
        raise
    except Exception as e:
        raise _http_error_from_transport(e, current_user.email)


@router.post("/translate/ensemble", response_model=EnsembleResponse)
async def translate_ensemble(
    request: EnsembleRequest,
    current_user: User = Depends(get_current_user),
):
    models = request.models or settings.ensemble_models_list

    if not is_ensemble_language_pair(request.source_lang, request.target_lang):
        raise HTTPException(
            status_code=400,
            detail=(
                "Ensemble translation is only available for languages "
                "supported by Ollama. Please use /translate endpoint for "
                "fallback."
            ),
        )

    if not models:
        raise HTTPException(status_code=400, detail="No models specified")

    try:
        mode = settings.EVAL_MODE
        glossary_entries, tm_entries = await collect_retrieval_context(
            request.text,
            request.source_lang,
            request.target_lang,
            mode,
        )

        translation_tasks = [
            _bounded_translate_with_model(
                request.text,
                request.source_lang,
                request.target_lang,
                model,
                glossary_entries,
                tm_entries,
                request.domain,
            )
            for model in models
        ]
        translation_outcomes = await asyncio.gather(
            *translation_tasks, return_exceptions=True
        )

        critic = Critic(engine=settings.AGENT_ENGINE)
        variants: List[EnsembleTranslationVariant] = []
        for model, outcome in zip(models, translation_outcomes):
            if isinstance(outcome, Exception):
                logger.error(
                    "Ensemble: model %s failed: %s", model, outcome, exc_info=True,
                )
                continue

            critic_state = {
                "source_text": request.text,
                "source_lang": request.source_lang,
                "target_lang": request.target_lang,
                "translation": outcome,
                "glossary_used": glossary_entries,
                "tm_used": tm_entries,
                "domain": request.domain,
                "audience": request.audience,
            }
            critic_state = await critic.process(critic_state)

            variants.append(
                EnsembleTranslationVariant(
                    model=model,
                    translation=outcome,
                    critic_score=critic_state.get("critic_score"),
                    critic_adequacy=critic_state.get("critic_adequacy"),
                    critic_fluency=critic_state.get("critic_fluency"),
                    critic_terminology=critic_state.get("critic_terminology"),
                    critic_reasoning=critic_state.get("critic_reasoning"),
                )
            )

        if not variants:
            raise HTTPException(
                status_code=502,
                detail="All ensemble models failed to produce a translation",
            )

        best_variant = select_best_variant(variants)

        best_clean = clean_translation(best_variant.translation)
        if best_clean and len(best_clean) > 1:
            best_translation = best_clean
        else:
            best_translation = best_variant.translation

        if request.domain == CACHEABLE_DOMAIN:
            await cache_manager.save_cache(
                request.text,
                request.source_lang,
                request.target_lang,
                best_translation,
                best_variant.model,
                engine="ollama",
                eval_mode=mode,
            )

        return EnsembleResponse(
            source_text=request.text,
            source_lang=request.source_lang,
            target_lang=request.target_lang,
            best_translation=best_translation,
            best_model=best_variant.model,
            best_critic_score=best_variant.critic_score or 0.0,
            variants=variants,
        )
    except RUNGException:
        raise
    except HTTPException:
        raise
    except Exception as e:
        raise _http_error_from_transport(e, current_user.email)
```

### Разбор ключевых решений

**Почему `_MAX_CONCURRENT_TRANSLATIONS = 4`.** Комментарий в коде: «Cap concurrent outbound LLM calls per request. A single `/translate/batch` with 50 texts would otherwise open 50 simultaneous connections to Ollama and either time out or exhaust the connection pool. 4 matches the practical parallelism of `OLLAMA_NUM_PARALLEL` on the deployed container.» Семафор — 4. `OLLAMA_NUM_PARALLEL` в compose (Этап 8) — `2`. Разница: semaphore ограничивает **исходящие** запросы из FastAPI, `OLLAMA_NUM_PARALLEL` — **одновременные** inference на стороне Ollama. 4 запроса от FastAPI, 2 обрабатываются, 2 в очереди Ollama. Компромисс: не «заваливаем» Ollama, но и не «сериализуем» всё в 1 поток.

**Почему `_translation_semaphore = asyncio.Semaphore(4)` на module-level.** Один на процесс. Все `/translate/batch` запросы **конкурируют** за одни и те же 4 слота. Если 5 батчей по 50 текстов одновременно — 250 задач, но только 4 идут в LLM, остальные ждут. Это правильно: общая пропускная способность к Ollama ограничена 4, независимо от количества клиентов.

**Почему `_http_error_from_transport` — отдельная функция.** Разворачивает 4 типа `httpx`-исключений в правильные HTTP-статусы. Без неё каждый роут имел бы копипасту:
```python
except (httpx.TimeoutException, httpx.ReadTimeout):
    raise HTTPException(504, "Upstream timeout")
except httpx.ConnectError:
    raise HTTPException(502, "Upstream service unavailable")
except httpx.HTTPStatusError as e:
    raise HTTPException(502, f"Upstream error: {e.response.status_code}")
except Exception as e:
    raise HTTPException(500, str(e))
```

4 роута × 8 строк = 32 строки копипасты. Одна функция — 20 строк. DRY.

**Почему `except RUNGException: raise` — первым.** `RUNGException` (Этап 2) имеет глобальный хендлер в `main.py` — возвращает JSON с `code`/`context`. Если роут поймает `RUNGException` и переделает в `HTTPException(500)` — потеряем `code`. `raise` без обработки — правильно.

**Почему `except HTTPException: raise` — вторым.** `HTTPException` от `perform_translation` (например, `UnknownEvalModeError` уже обёрнут через RUNGException, но `HTTPException` тоже может быть). Пробрасываем как есть — FastAPI-хендлер развернёт в `{"detail": ...}`.

**Почему `except Exception: raise _http_error_from_transport(e, ...)` — последним.** Ловит всё остальное: `httpx`, `asyncio.TimeoutError`, `RuntimeError` из pipeline. Преобразует в `HTTPException` с правильным статусом. Пользователь видит `502 Upstream service unavailable`, а не `500 Internal Server Error` с трейсбеком.

**Почему `_build_details` — отдельная функция.** Используется в `/translate/multi` (для `details: Dict[str, TranslationDetails]`) и `/translate/batch` (для `details: List[TranslationDetails]`). `TranslationDetails` — Pydantic-модель, поля: `glossary_used`, `tm_used`, `critic_score`, `validation_passed`, `critic_adequacy`, `critic_fluency`, `critic_terminology`, `critic_suggestions`, `critic_missing_terms`, `reasoning_log`, `engine_used`. Не включает `translation`, `model_used` (они в корне `TranslateMultiResponse`/`TranslateBatchResponse`).

**Почему `result.get("reasoning_log")` в `_build_details`, а `result["glossary_used"]` в других.** `empty_result` (кэш-hit, ошибка) содержит `reasoning_log: []` — есть. Но на случай, если `result` пришёл от `safe_perform_translation` с какой-то другой структурой — `get` безопаснее. Все остальные поля (`glossary_used`, `tm_used`, ...) гарантированно есть в `perform_translation` и `empty_result`. У всех `KeyError`-риск минимален, но для `reasoning_log` — `.get`.

**Почему `_bounded_translate` **не** используется в `/translate`.** `/translate` — один текст. Семафор на 4 не нужен (сам запрос один). `_bounded_translate` даёт `(ok, result)` — двойная работа в single-case. Прямой `perform_translation` — правильно. Семафор применятся только в `/multi` и `/batch`, где задачи конкурируют.

**Почему `_bounded_translate` оборачивает `safe_perform_translation`, а не `perform_translation`.** `safe_` ловит исключения внутри и возвращает `(False, placeholder)`. Если бы оборачивали `perform_translation` — при исключении `gather` получил бы `Exception`, `zip`/`for` упал бы на unpack. `safe_` даёт предсказуемую структуру `(bool, dict)` для каждого outcome.

**Почему `zip(request.target_langs, outcomes)` в `/multi`.** `asyncio.gather(*tasks)` сохраняет порядок: `outcomes[i]` соответствует `tasks[i]` соответствует `request.target_langs[i]`. `zip` собирает пары. Внутри: `for target_lang, (ok, result) in zip(...)` — `ok` не используется в `/multi` (failures не считаются), но `_bounded_translate` возвращает tuple — нужен для распаковки.

**Почему `/multi` не считает failures.** В отличие от `/batch`, `/multi` возвращает `translations: Dict[str, str]` — по языку. Если для языка упал перевод, `result["translation"] = ""` — клиент видит пустоту для этого языка. `details[target_lang]` — заполнены, `critic_reasoning = "Translation failed: ..."`. Клиент может распознать failure по `not translation`. В `/batch` считается `failures` для лога (но не влияет на ответ).

**Почему `/batch` — `_bounded_translate` с тем же `source_lang`/`target_lang`, но разными `text`.** Каждый текст идёт через `safe_perform_translation` под семафором. Если один текст даёт timeout — `(False, placeholder)`, остальные продолжают. `translations.append(result["translation"])` — `""` для упавших. `details.append(_build_details(result))` — `critic_reasoning` содержит ошибку.

**Почему `/translate/ensemble` **не** использует `_bounded_translate`.** Ensemble — не `perform_translation`, а `translate_with_model` (прямой вызов LLM без кэша, retrieval, агентов). `_bounded_translate_with_model` — отдельная функция, оборачивающая `translate_with_model`. Семафор — тот же, чтобы ensemble не «съел» все слоты.

**Почему `return_exceptions=True` в `asyncio.gather(*translation_tasks, return_exceptions=True)` для ensemble.** Если одна модель timeout — не хочется, чтобы `gather` упал целиком. `return_exceptions=True` — каждая задача завершается `Translation` или `Exception`. В цикле `if isinstance(outcome, Exception): continue` — упавшие пропускаются.

**Почему для ensemble Critic создаётся **один** раз (`critic = Critic(engine=settings.AGENT_ENGINE)`), а не на каждый вариант.** `Critic.__init__` — лёгкий (только `get_hf_llm` или `None`). Переиспользование — меньше overhead. `Critic.process` — stateless (каждый вызов обновляет `_last_result`, но это не влияет на следующий).

**Почему `critic_state` для ensemble — минимальный (без `use_critic`/`use_editor`/`engine`).** `Critic.process` (Этап 5) читает: `source_text`, `translation`, `source_lang`, `target_lang`, `glossary_used`, `reference`, `domain`, `audience`. Не читает `use_critic`, `engine`. Минимальный state — правильно. `Supervisor` тут не нужен (нет цикла Editor).

**Почему `if not variants: 502`.** Все модели упали с `Exception` → `variants = []`. Нечего выбирать. 502 Bad Gateway — «upstream (модели) недоступны». `detail="All ensemble models failed..."`.

**Почему `clean_translation(best_variant.translation)` в ensemble.** `translate_with_model` возвращает результат `call_ollama_with_model` — уже **чистый** (потому что `call_ollama_with_model` делает `clean_translation` внутри). Но на всякий случай — двойная очистка не вредит. Если `best_clean` пуст или 1 символ — `best_translation = best_variant.translation` (raw). Тот же паттерн, что в `perform_translation`.

**Почему в ensemble `engine="ollama"` — hardcoded.** Ensemble доступен только для `KNOWN_LANGUAGES`, где `translator_manager` всегда выберет Ollama (если Ollama доступна). HF-path не вызывается. `engine="ollama"` в кэш — правильно.

**Почему `mode = settings.EVAL_MODE` в ensemble.** Ensemble не принимает `eval_mode` из запроса (нет поля в `EnsembleRequest`). Берёт из `.env`. `collect_retrieval_context` использует `mode` для `_retrieve_context`. `save_cache` — с `eval_mode=mode`. Правильно: ensemble → один глобальный режим.

**Почему `best_critic_score=best_variant.critic_score or 0.0`.** `critic_score: Optional[float]`. Если None (Critic упал для лучшего варианта) → `0.0`. `EnsembleResponse.best_critic_score: float` — не Optional. `or 0.0` — fallback.

**Почему `select_best_variant` **не** вызывает `clean_translation`.** `select_best_variant` — чистая функция выбора по score. `clean_translation` — в `translate_ensemble` после выбора. Разделение: `pipeline.select_best_variant` — чистая логика, тестируемая без LLM. `translate.py` — оркестрация, чистит и кэширует.

---

## 9.5. `backend/api/routes/admin.py` — CRUD и CSV

**Что это:** `/admin/*` — управление glossary/TM/cache. Все под `require_admin` (Этап 4). `_import_entries_from_csv` — общий для glossary и TM.

**Зачем это нужно:** админ должен добавлять/удалять термины и TM-пары через API (Streamlit, Этап 12). Импорт CSV — для массовой загрузки (Этап 6 — скрипты, но веб-интерфейс удобнее).

Создайте `backend/api/routes/admin.py`:

```python
# backend/api/routes/admin.py
import csv
import io
import logging
from typing import Any, Dict, List, Optional, Type

from fastapi import APIRouter, Depends, File, HTTPException, Query, UploadFile, status
from sqlalchemy import func, select
from sqlalchemy.ext.asyncio import AsyncSession

from backend.api.deps import require_admin
from backend.core.exceptions import ImportFormatError, RUNGException
from backend.core.models import GlossaryEntry, TMEntry
from backend.data.cache import cache_manager
from backend.data.glossary import glossary_manager
from backend.data.translation_memory import tm_manager
from backend.db.models import TranslationCache
from backend.db.session import get_async_session

logger = logging.getLogger(__name__)
router = APIRouter(prefix="/admin", tags=["admin"])

_REQUIRED_CSV_COLUMNS = {"source_lang", "target_lang"}
_LEGACY_TEXT_COLUMNS = {"en", "tgt"}


async def _import_entries_from_csv(
    file: UploadFile,
    entry_cls: Type,
    add_entry,
) -> Dict[str, Any]:
    if not file.filename or not file.filename.lower().endswith(".csv"):
        raise ImportFormatError("File must be CSV")

    content = await file.read()
    try:
        text = content.decode("utf-8-sig")
    except UnicodeDecodeError:
        raise ImportFormatError("File must be UTF-8 encoded")

    reader = csv.DictReader(io.StringIO(text))
    fieldnames = set(reader.fieldnames or [])

    if not _REQUIRED_CSV_COLUMNS.issubset(fieldnames):
        raise ImportFormatError(
            f"CSV must contain columns: {sorted(_REQUIRED_CSV_COLUMNS)}"
        )

    has_legacy_columns = _LEGACY_TEXT_COLUMNS.issubset(fieldnames)

    if not has_legacy_columns:
        lang_columns = fieldnames - _REQUIRED_CSV_COLUMNS
        if not lang_columns:
            raise ImportFormatError(
                "CSV must contain either 'en'/'tgt' columns, or "
                "columns named after the source/target language codes"
            )

    added = 0
    skipped = 0
    errors: List[str] = []

    for row_num, row in enumerate(reader, start=2):
        source_lang = (row.get("source_lang") or "").strip()
        target_lang = (row.get("target_lang") or "").strip()

        if not source_lang or not target_lang:
            errors.append(f"Row {row_num}: missing source_lang/target_lang")
            skipped += 1
            continue

        if has_legacy_columns:
            en = (row.get("en") or "").strip()
            tgt = (row.get("tgt") or "").strip()
        else:
            if source_lang not in fieldnames:
                errors.append(
                    f"Row {row_num}: header has no '{source_lang}' column"
                )
                skipped += 1
                continue
            if target_lang not in fieldnames:
                errors.append(
                    f"Row {row_num}: header has no '{target_lang}' column"
                )
                skipped += 1
                continue
            en = (row.get(source_lang) or "").strip()
            tgt = (row.get(target_lang) or "").strip()

        if not en or not tgt:
            errors.append(f"Row {row_num}: empty source or target text")
            skipped += 1
            continue

        try:
            entry = entry_cls(
                en=en, tgt=tgt,
                source_lang=source_lang, target_lang=target_lang,
            )
            await add_entry(entry)
            added += 1
        except Exception as e:
            errors.append(f"Row {row_num}: {e}")
            skipped += 1

    return {
        "status": "ok",
        "added": added,
        "skipped": skipped,
        "errors": errors[:20],
    }


@router.post("/glossary/add", status_code=status.HTTP_201_CREATED)
async def add_glossary_entry(
    entry: GlossaryEntry,
    _=Depends(require_admin),
):
    try:
        await glossary_manager.add_entry(entry)
        return {"status": "ok", "message": f"Added: {entry.en} → {entry.tgt}"}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to add glossary entry: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.post("/glossary/import-csv", status_code=status.HTTP_201_CREATED)
async def import_glossary_csv(
    file: UploadFile = File(..., description=(
        "CSV with columns: source_lang,target_lang,en,tgt "
        "or source_lang,target_lang,<src>,<tgt>"
    )),
    _=Depends(require_admin),
):
    try:
        return await _import_entries_from_csv(
            file, GlossaryEntry, glossary_manager.add_entry
        )
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Glossary CSV import failed: %s", e)
        raise HTTPException(status_code=500, detail=f"Import failed: {e}")


@router.get("/glossary/list")
async def list_glossary(
    source_lang: Optional[str] = None,
    target_lang: Optional[str] = None,
    limit: int = Query(100, ge=1, le=1000),
    offset: int = Query(0, ge=0),
    _=Depends(require_admin),
):
    try:
        entries = await glossary_manager.list_all(
            source_lang, target_lang, limit=limit, offset=offset,
        )
        total = await glossary_manager.count(source_lang, target_lang)
        return {"entries": entries, "total": total, "limit": limit, "offset": offset}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to list glossary: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.delete("/glossary/delete/{entry_id}")
async def delete_glossary_entry(entry_id: str, _=Depends(require_admin)):
    try:
        await glossary_manager.delete_entry(entry_id)
        return {"status": "ok", "message": "Deleted"}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to delete glossary entry: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.post("/tm/add", status_code=status.HTTP_201_CREATED)
async def add_tm_entry(
    entry: TMEntry,
    _=Depends(require_admin),
):
    try:
        await tm_manager.add_entry(entry)
        return {"status": "ok", "message": f"Added: {entry.en} → {entry.tgt}"}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to add TM entry: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.post("/tm/import-csv", status_code=status.HTTP_201_CREATED)
async def import_tm_csv(
    file: UploadFile = File(..., description=(
        "CSV with columns: source_lang,target_lang,en,tgt "
        "or source_lang,target_lang,<src>,<tgt>"
    )),
    _=Depends(require_admin),
):
    try:
        return await _import_entries_from_csv(
            file, TMEntry, tm_manager.add_entry
        )
    except RUNGException:
        raise
    except Exception as e:
        logger.error("TM CSV import failed: %s", e)
        raise HTTPException(status_code=500, detail=f"Import failed: {e}")


@router.get("/tm/list")
async def list_tm(
    source_lang: Optional[str] = None,
    target_lang: Optional[str] = None,
    limit: int = Query(100, ge=1, le=1000),
    offset: int = Query(0, ge=0),
    _=Depends(require_admin),
):
    try:
        entries = await tm_manager.list_all(
            source_lang, target_lang, limit=limit, offset=offset,
        )
        total = await tm_manager.count(source_lang, target_lang)
        return {"entries": entries, "total": total, "limit": limit, "offset": offset}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to list TM: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.delete("/tm/delete/{entry_id}")
async def delete_tm_entry(entry_id: str, _=Depends(require_admin)):
    try:
        await tm_manager.delete_entry(entry_id)
        return {"status": "ok", "message": "Deleted"}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to delete TM entry: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.delete("/cache/clear")
async def clear_cache(
    source_lang: Optional[str] = None,
    target_lang: Optional[str] = None,
    eval_mode: Optional[str] = None,
    _=Depends(require_admin),
):
    try:
        deleted = await cache_manager.clear_cache(source_lang, target_lang, eval_mode)
        return {"status": "ok", "deleted": deleted}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to clear cache: %s", e)
        raise HTTPException(status_code=500, detail=str(e))


@router.get("/cache/stats")
async def cache_stats(
    session: AsyncSession = Depends(get_async_session),
    _=Depends(require_admin),
):
    try:
        total = await session.scalar(
            select(func.count()).select_from(TranslationCache)
        )
        total_hits = await session.scalar(
            select(func.sum(TranslationCache.hits))
        ) or 0
        return {"total_entries": total, "total_hits": total_hits}
    except RUNGException:
        raise
    except Exception as e:
        logger.error("Failed to get cache stats: %s", e)
        raise HTTPException(status_code=500, detail=str(e))
```

### Разбор ключевых решений

**Почему `_import_entries_from_csv` — generic через `entry_cls` и `add_entry`.** `GlossaryEntry` и `TMEntry` имеют одинаковые поля (`en`, `tgt`, `source_lang`, `target_lang`). `glossary_manager.add_entry` и `tm_manager.add_entry` — одинаковый контракт (`await add_entry(entry)`). Один код для обоих — DRY. `entry_cls: Type` — `GlossaryEntry` или `TMEntry`. `add_entry` — bound method.

**Почему `_REQUIRED_CSV_COLUMNS = {"source_lang", "target_lang"}` — module-level.** Валидация «обязательные колонки» — одинаковая для glossary и TM. `_LEGACY_TEXT_COLUMNS = {"en", "tgt"}` — тоже. Вынесены — переиспользуются в проверках.

**Почему проверка `filename.endswith(".csv")` — до чтения файла.** Быстрая валидация. Если файл называется `.txt` — не тратим время на `await file.read()` (может быть большой). `ImportFormatError` → 400 (Этап 2: `ImportFormatError` наследует `KnowledgeBaseError` → 400).

**Почему `content.decode("utf-8-sig")`.** Excel сохраняет CSV с BOM (`\ufeff`). Без `-sig` первая колонка называется `"\ufeffsource_lang"` — `issubset` не найдёт `source_lang`. `utf-8-sig` удаляет BOM.

**Почему два формата CSV — legacy и lang-coded.** Legacy: `source_lang,target_lang,en,tgt` — фиксированные имена колонок для текстов. Lang-coded: `source_lang,target_lang,ar,ru` — колонки названы кодами языков, соответствует `source_lang`/`target_lang` в строке. Lang-coded удобнее для файлов, где каждая строка — одна пара. Legacy — для обратной совместимости (Этап 6, `add_glossary.py`).

**Почему `if not has_legacy_columns: lang_columns = fieldnames - REQUIRED`.** Если нет `en`/`tgt`, значит lang-coded — нужны колонки, названные кодами. `fieldnames - {"source_lang", "target_lang"}` — все остальные. Если пусто — CSV только с `source_lang,target_lang`, без текстов — ошибка.

**Почему в lang-coded проверяем `source_lang not in fieldnames` для **каждой** строки.** В одной строке `source_lang="ar"`, в другой — `"ru"`. Header содержит `ar,ru,en,tgt` (например). Для строки с `ar` — колонка `ar` есть. Для `ru` — есть. Но если в файле только `ar`-колонки, а строка с `ru` — ошибка «header has no 'ru' column». Плюс в начале общая проверка `lang_columns` — если ни одной лишней колонки, файл точно невалиден.

**Почему `enumerate(reader, start=2)`.** Номер строки для лога. Header — строка 1, data — со 2-й. Пользователь ищет в Excel по номеру — совпадёт.

**Почему `errors[:20]`.** Если файл на 10000 строк и все битые — отчёт в 10000 строк бесполезен. Первые 20 — достаточно, чтобы понять проблему. Полный список — в логах (`logger.warning`).

**Почему `await add_entry(entry)` — в try/except, `added += 1` только при успехе.** `GlossaryEntry(**...)` / `TMEntry(**...)` могут бросить `ValueError` (Pydantic-валидатор языка). `add_entry` может бросить `IntegrityError`/`OperationalError`. Всё ловим, добавляем в `errors`, `skipped += 1`.

**Почему `_import_entries_from_csv` не логирует `skipped` отдельно от `added`.** В ответе — `{added, skipped, errors}`. `skipped = сумма ошибочных строк + дубликаты`. Но `add_entry` для glossary не сообщает «это дубликат» (SELECT-then-INSERT return при существующем). Для TM — `add_many` возвращает `len(inserted)`, но мы вызываем `add_entry` (который внутри `add_many([entry])`). Если `inserted=0` — `add_entry` не бросил, `added += 1`. То есть `added` включает и дубликаты для glossary/TM. **Ограничение**: точный счётчик «реально новых» не даётся. Для UI-отчёта — приемлемо, для статистики — `SELECT COUNT(*)` после.

**Почему `POST /admin/glossary/add` — `entry: GlossaryEntry` в теле, а не `UploadFile`.** Одиночное добавление — JSON. `GlossaryEntry` — Pydantic, валидирует поля. Импорт CSV — отдельный endpoint.

**Почему `DELETE /admin/glossary/delete/{entry_id}` — path-параметр, а не query.** `entry_id` содержит `|` (SHA-256 формат: `<src>|<tgt>|<hash>`). В path `|` работает без URL-кодирования. В query — тоже, но семантически «удалить запись с id» — path.

**Почему `GET /admin/cache/stats` — `Depends(get_async_session)`, а не через `cache_manager`.** `cache_manager.get_cached`/`save_cache` — точечные операции с собственной сессией. Для `COUNT(*)` и `SUM(hits)` — прямой запрос. `get_async_session` (Этап 3) даёт сессию на время запроса. Одна сессия — два запроса (`count` + `sum`).

**Почему `total_hits = await session.scalar(...) or 0`.** Если таблица пуста — `SUM(hits)` вернёт `NULL`. `or 0` — `NULL` → `0`. Без `or 0` — `total_hits: None`, тип неправильный.

**Почему `limit: int = Query(100, ge=1, le=1000)`.** `Query` валидирует: `limit=0` → 422. `limit=10000` → 422. Защита от «запросить всё». Дефолт 100 — для UI.

**Почему все роуты под `_=Depends(require_admin)`.** `require_admin` (Этап 4) проверяет `current_user.role == "admin"`, иначе 403. `_` — FastAPI передаёт пользователя, но нам он не нужен (используется только для проверки). Стандартный паттерн.

**Почему `except RUNGException: raise`, потом `except Exception: 500`.** `glossary_manager.add_entry` может бросить `RUNGException`-наследника (например, `DuplicateEntryError`, хотя в Этапе 7 его нет). Пробрасываем. Остальные — 500 с логом.

**Почему `POST /admin/glossary/import-csv` возвращает `{status, added, skipped, errors}` — не 201 с телом.** `status_code=status.HTTP_201_CREATED` — «создан ресурс». Но тело — отчёт об импорте, не одна запись. 201 семантически «что-то создано» — ok. Тело — отчёт.

---

## 9.6. `backend/api/main.py` — регистрация роутеров

**Что это:** в `main.py` (создан в Этапе 2, обновлён в Этапах 3, 4, 8) добавить `app.include_router(translate.router)` и `app.include_router(admin.router)`.

**Зачем это нужно:** без `include_router` роуты не зарегистрируются, `/translate` и `/admin/*` вернут 404.

Откройте `backend/api/main.py`. Найдите блок `app.include_router(...)` (после Этапа 4 там `health.router` и `auth.router`), и обновите:

**1. Импорт — добавить `admin, translate`:**

```python
from backend.api.routes import admin, auth, health, translate
```

**2. После `app.include_router(auth.router)` — добавить:**

```python
app.include_router(translate.router)
app.include_router(admin.router)
```

Проверьте, что порядок такой (не важен для FastAPI, но для читаемости):

```python
app.include_router(health.router)
app.include_router(auth.router)
app.include_router(translate.router)
app.include_router(admin.router)
```

### Проверка

```powershell
python -m poetry run python -c "from backend.api.main import app; print(sorted([r.path for r in app.routes if hasattr(r, 'path')]))"
```

**Что вы должны увидеть** (все пути — `/translate`, `/translate/multi`, `/translate/batch`, `/translate/ensemble`, `/admin/...`):

```
['/admin/cache/clear', '/admin/cache/stats', '/admin/glossary/add', '/admin/glossary/delete/{entry_id}', '/admin/glossary/import-csv', '/admin/glossary/list', '/admin/tm/add', '/admin/tm/delete/{entry_id}', '/admin/tm/import-csv', '/admin/tm/list', '/auth/login', '/auth/me', '/auth/register', '/health', '/healthz', '/languages', '/translate', '/translate/batch', '/translate/ensemble', '/translate/multi', ...]
```

---

## 9.7. Тесты

Тесты скинет пользователь. Здесь — что должно проверяться.

### `tests/unit/test_pipeline.py`

- `_mode_config("full")` → dict с `use_glossary=True, tm_backend="hybrid", use_critic=True, use_editor=True`.
- `_mode_config("unknown")` → `UnknownEvalModeError`.
- `build_translation_prompt` содержит `Glossary:` при `glossary_entries` не пустом.
- `build_translation_prompt` содержит `(score: 0.85)` для `TMEntry` со score.
- `build_translation_prompt` **не** содержит `(score: ...)` для `TMEntry` без score.
- `build_translation_prompt` содержит `Domain: medical` при `domain="medical"`, не содержит при `"general"`.
- `select_best_variant` все `critic_score=None` → первый, warning в логе.
- `select_best_variant` один с max score → он.
- `select_best_variant` tie → длиннее.
- `empty_result(engine="ollama", translator="cache", model="m")` — все поля на месте.
- `perform_translation` с мок `translator_manager` → Ollama, `engine_used="ollama"`.
- `perform_translation` с `engine="ollama"`, `translator_manager` → HF, `strict_engine=True` → `EngineFallbackTriggeredError`.
- `perform_translation` арабский источник → `strip_arabic_diacritics` вызван.
- `safe_perform_translation` при исключении → `(False, dict)` с `translator="error"`.
- `is_ensemble_language_pair("en", "ru")` → True; `("en", "udm")` → False.
- `_retrieve_context` для `baseline` → `glossary_entries=[], tm_entries=[]` (не вызывает ничего).
- `_retrieve_context` для `tm` → вызывает `tm_manager.search`.
- `_retrieve_context` для `tm_hybrid` → вызывает `hybrid_search`.

**Mock стратегия.** `monkeypatch.setattr("backend.services.pipeline.translator_manager", MockTranslatorManager())`, `monkeypatch.setattr("backend.services.pipeline.Supervisor", MockSupervisor)`, `monkeypatch.setattr("backend.services.pipeline.glossary_manager.get_entries", AsyncMock(return_value=[]))` и т.д. Плюс `monkeypatch.setattr("backend.services.pipeline.cache_manager.get_cached", AsyncMock(return_value=None))`.

### `tests/unit/test_cache.py`

- `_normalize(None)` → `"unknown"`, `_normalize("")` → `"unknown"`, `_normalize("  ")` → `"unknown"`.
- `save_cache` с `engine=None` → записано с `engine_used="unknown"`.
- `get_cached(engine=None)` — не фильтрует по engine (мок SQL — проверить, что `WHERE` не содержит `engine_used`).
- `get_cached` инкрементит `hits`.
- `save_cache` с существующей записью → no-op (без второй INSERT).
- `clear_cache()` → все.
- `clear_cache(source_lang="en")` → только en.
- `clear_cache(eval_mode="tm_hybrid")` → только tm_hybrid.
- `get_cached` при `OperationalError` → `None` (не raise).

### `tests/integration/test_api.py` — 8 режимов и кэш

- `POST /translate` с мок-LLM, `mode=baseline` → 200, `critic_score=None`, `validation_passed` заполнено (Validator отработал).
- `POST /translate` с `mode=glossary` → глоссарий вызван (`mock_glossary.get_entries` — assert called).
- `POST /translate` с `mode=tm_hybrid` → `hybrid_search` вызван.
- `POST /translate` с `mode=full` → Supervisor вызван, `critic_score` в ответе.
- `POST /translate` 2 раза подряд с `engine=None` → 2-й раз кэш-hit (`translator_used="cache"`).
- `POST /translate` с `engine="ollama"`, `translator_manager` возвращает HF, `strict_engine=True` (в pipeline) → 503.
- `POST /translate/multi` с `target_langs=["ru", "tt"]` → `translations: {"ru": "...", "tt": "..."}`, `details` для обоих.
- `POST /translate/batch` с `texts=["a", "b"]` → `translations: [...]` длины 2.
- `POST /translate/ensemble` с `source_lang="en", target_lang="ru"` → 200, `variants`, `best_translation`.
- `POST /translate/ensemble` с `source_lang="en", target_lang="udm"` → 400 (udm не в `KNOWN_LANGUAGES`).
- `POST /translate/ensemble` с `models=[]` → 400.
- `POST /translate/ensemble` с одной моделью timeout (мок) → 200, `variants` без упавшей, `best_*` есть.
- `POST /translate/ensemble` все модели timeout → 502.
- `POST /translate` без токена → 401.
- `POST /admin/glossary/add` под `user` → 403.
- `POST /admin/glossary/add` под `admin` → 201.
- `POST /admin/glossary/import-csv` под `admin`, CSV с 3 строками → `{added: 3, skipped: 0, errors: []}`.
- `POST /admin/glossary/import-csv` повторно → `{added: 3, skipped: 0}` (или `added: 3` — `add_entry` не считает дубликаты).
- `POST /admin/tm/import-csv` повторно → `{added: 0, skipped: N}` (TM использует `add_many`, возвращает реально вставленное).

**Mock LLM.** `monkeypatch.setattr("backend.utils.llm_client.call_ollama_with_model", AsyncMock(return_value="Перевод"))`. `monkeypatch.setattr("backend.services.pipeline.call_ollama_with_model", ...)` — для `perform_translation`. `monkeypatch.setattr("backend.agents.critic.Critic._call_llm", AsyncMock(return_value='{"adequacy": 8, ...}'))`.

**Mock TranslatorManager.** `monkeypatch.setattr("backend.services.pipeline.translator_manager.get_translator_for_engine", AsyncMock(return_value=None))` — Ollama. Для HF-сценария — `AsyncMock(return_value=mock_hf_translate_func)`.

**Mock Qdrant.** `monkeypatch.setattr("backend.retrieval.vector_index.get_vector_index", lambda: MockVectorIndex())` — `MockVectorIndex.add_entries`, `.search`, `.recreate_collection` — no-op или `return []`.

### `tests/integration/test_cache_flow.py`

- `POST /translate` → 200, кэш сохранён (`SELECT COUNT(*) FROM translation_cache` = 1).
- Повторный `POST /translate` → `translator_used="cache"`, `hits` = 2 (проверить `SELECT hits FROM translation_cache`).
- `POST /translate` с `domain="medical"` → кэш **не** сохранён (`COUNT` не растёт).
- `DELETE /admin/cache/clear` → `deleted: 1`.

**Фикстура `autouse` для `unload_hf_llm`.** В `conftest.py`:
```python
@pytest.fixture(autouse=True)
def clean_hf_models():
    yield
    from backend.utils.hf_llm import unload_hf_llm
    unload_hf_llm()
```

---

## 9.8. Финальная проверка

```powershell
python -m poetry run pytest tests/ -v
```

Полный прогон всех тестов Этапов 1–9. Все зелёные.

Финальный чек-лист:

- [ ] `curl http://localhost:8000/health` → 200, все `services.*.healthy: true`.
- [ ] `curl -X POST http://localhost:8000/translate -H "Authorization: Bearer <token>" -H "Content-Type: application/json" -d "{\"text\": \"Hello\", \"source_lang\": \"en\", \"target_lang\": \"ru\"}"` → 200 с `translation`.
- [ ] `curl -X POST http://localhost:8000/translate ... -d "{\"text\": \"Hello\", \"source_lang\": \"en\", \"target_lang\": \"udm\"}"` → 200, `engine_used: "huggingface"` (udm не в KNOWN).
- [ ] `curl -X POST http://localhost:8000/translate/ensemble ... -d "{\"text\": \"Hello\", \"source_lang\": \"en\", \"target_lang\": \"ru\", \"models\": [\"qwen2.5:7b\"]}"` → 200, `variants`, `best_translation`.
- [ ] `curl -X POST http://localhost:8000/translate/ensemble ... -d "{\"source_lang\": \"en\", \"target_lang\": \"udm\", ...}"` → 400.
- [ ] `curl -X POST http://localhost:8000/admin/glossary/add -H "Authorization: Bearer <admin_token>" ...` → 201.
- [ ] `curl -X POST http://localhost:8000/admin/glossary/add -H "Authorization: Bearer <user_token>" ...` → 403.
- [ ] Повторный `/translate` → `translator_used: "cache"`.
- [ ] `pytest` — зелёный.

Коммит и merge:

```powershell
git add -A
git commit -m "feat(week-9): complete pipeline, translate routes, admin CRUD, cache"
git push -u origin week-9
```

---

## Что дальше

В **Этапе 10** — HITL: `ActiveLearningManager` (уже создан в Этапе 10 в `backend/hitl/active_learning.py` — используется в 9.1 через `add_uncertain_example`). Роуты `/hitl/*` (созданы в Этапе 10, но публикуются в 10). Связка: `perform_translation` при `critic_score < 7` уже пишет в `uncertain_examples` (9.1), в Этапе 10 — читаем и подтверждаем.

**В Этапе 11** — Celery worker + beat. `redis` уже поднят (Этап 8). `index_translation_memory`, `batch_translate`, `clear_old_cache` — задачи.

**В Этапе 12** — Streamlit UI (8 страниц), скрипты eval (`run_eval.py`, `paired_bootstrap.py`, `report_tables.py`), README.

Модуль `backend/services/pipeline.py` больше не меняется — все дальнейшие этапы его используют.